# Spatial Copilot — notebook loader

The notebook twin of `main.py`. Same options, same code: this file **imports `main.py`**
instead of re-implementing it, so the two can't drift apart.

| Option | What it does |
|---|---|
| **check** | Python, packages, CUDA/MPS, dataset + weights on disk |
| **train** | YOLO11-seg fine-tune with a live per-epoch progress bar |
| **load** | Floorplan segmenter (ResNet34-UNet) or a YOLO checkpoint, on one image |
| **serve** | FastAPI + WebXR viewer (`/health`, `/docs`, `/static/viewer.html`) |
| **download** | Pretrained segmenter weights → `weights/` |

Works locally (use the repo venv kernel) and in Colab (cell 1 clones + optional install).

---
### 1 — Bootstrap: find the repo, import `main`, install on Colab

In [ ]:
# @title Bootstrap (run me first)
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/joshyajith863/Spatial_model.git"
INSTALL_APP_DEPS = False   # Colab only: install the web/3D stack too

if IN_COLAB:
    if not Path("main.py").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
        os.chdir("Spatial_model")
    if INSTALL_APP_DEPS:
        # NOT `pip install -r requirements.txt`: it pins CPU torch and would
        # replace Colab's CUDA build. These are the app-only extras.
        deps = ["fastapi", "uvicorn", "shapely", "trimesh", "mapbox-earcut",
                "ezdxf", "opencv-python-headless", "pymupdf", "pyyaml",
                "safetensors", "segmentation-models-pytorch"]
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *deps],
                       check=False)


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up until we find main.py next to app/ (notebook may live in a subdir)."""
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "main.py").exists() and (candidate / "app").is_dir():
            return candidate
    return here


ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import main as loader  # noqa: E402  (same functions the CLI uses)

print(f"repo     : {ROOT}")
print(f"python   : {sys.version.split()[0]}")
print(f"colab    : {IN_COLAB}")
print(f"loader   : {loader.__file__}")

---
### 2 — Menu (same five options as `python main.py`)

Pick an entry and press **Run**. Options that need input (`train` epochs, a checkpoint
path, ...) prompt underneath the button — the same prompts the CLI asks.

In [ ]:
# @title Interactive menu (ipywidgets; falls back to text)
ACTIONS = {
    "check     - environment, weights, dataset": loader.system_check,
    "train     - YOLO11-seg fine-tune (progress bar)": loader.interactive_train,
    "load      - segmenter / YOLO on an image": loader.interactive_load,
    "serve     - API + WebXR viewer": loader.interactive_serve,
    "download  - segmenter weights": loader.download_weights,
}

try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    choice = widgets.Dropdown(options=list(ACTIONS), description="action",
                              layout=widgets.Layout(width="520px"))
    run = widgets.Button(description="Run", button_style="primary")
    log = widgets.Output()

    def _on_run(_button=None):
        with log:
            clear_output()
            print(f"▶ {choice.value}", flush=True)
            ACTIONS[choice.value]()
            print("done.", flush=True)

    run.on_click(_on_run)
    display(widgets.HBox([choice, run]), log)

except ImportError:  # plain-fallback
    print("ipywidgets not installed - call one directly, e.g.:\n")
    for i, (label, fn) in enumerate(ACTIONS.items(), 1):
        print(f"  {i}) {label}")
    print("\n  loader.system_check()")

---
### 3 — Environment check

No prompts. Tells you what's missing before you spend an hour on anything else.

In [ ]:
loader.system_check()

---
### 4 — Train (explicit parameters, live progress bar)

Set the knobs, run the cell. `--limit` builds a subset for a smoke run; `device="auto"`
uses CUDA/MPS when present, else CPU.

| knob | notes |
|---|---|
| `DEVICE` | `"auto"`, `"cpu"`, or `"0"` for the first GPU |
| `LIMIT` | images per split; `200` for a smoke run, `0` for everything |
| `PROJECT` | run output dir. In Colab point this at Drive so weights survive |
| `SAVE_PERIOD` | keep `epochN.pt` every N epochs; `-1` = only `last.pt`/`best.pt` |

Every epoch writes `weights/last.pt`; the next cell resumes from it after a disconnect.

Needs `data/yolo/data.yaml` (build it with `python scripts/convert_cubicasa_yolo.py`).

In [ ]:
EPOCHS = 2          # 50+ for a real run
BATCH = 16          # 8 on a small GPU / CPU
IMGSZ = 640
DEVICE = "auto"     # "0" to force the first GPU
LIMIT = 200         # 0 = full 4,228-image train split
NAME = "notebook_smoke"
PROJECT = "runs/segment"   # Colab: "/content/drive/MyDrive/spatial_runs/segment"
SAVE_PERIOD = 5     # also keep epoch5.pt, epoch10.pt... (-1 = only last/best)

# LIMIT>0 builds a real subset (Ultralytics has no image cap); 0 = everything.
data_arg = loader.build_subset_dataset(loader.DEFAULT_DATA, LIMIT) if LIMIT else loader.DEFAULT_DATA

loader.train_model(data=Path(data_arg), epochs=EPOCHS, batch=BATCH, imgsz=IMGSZ,
                   device=DEVICE, project=PROJECT, name=NAME, exist_ok=True,
                   save_period=SAVE_PERIOD)


### 4b — Resume an interrupted run

Ultralytics writes `weights/last.pt` at the end of every epoch, so an interrupt costs
one epoch at most. `train_model(resume=...)` continues in the checkpoint's own run
directory (its project/name/epochs are read from the checkpoint, not from the knobs above).


In [ ]:
# Nothing to do when no checkpoint exists - safe to run any time.
from pathlib import Path

run_dir = Path(PROJECT) / NAME
ckpt = run_dir / "weights" / "last.pt"

if ckpt.exists():
    print(f"resuming from {ckpt} ({ckpt.stat().st_size / 1e6:.1f} MB)")
    loader.train_model(resume=ckpt, device=DEVICE)
else:
    print("no checkpoint at", ckpt)
    print("run the training cell above first")


---
### 5 — Load models on one image

Segmenter first (pure CPU, no fine-tuning needed), then YOLO if you have a checkpoint.
Set `IMAGE = None` to use the first validation plan.

In [ ]:
IMAGE = None          # e.g. "data/yolo/images/valid/<file>.png"
CONF = 0.25

if IMAGE is None:
    candidates = sorted((ROOT / "data" / "yolo" / "images" / "valid").glob("*.png"))
    IMAGE = str(candidates[0]) if candidates else None
print(f"image: {IMAGE}")

loader.load_segmenter(image=IMAGE)

In [ ]:
# YOLO: auto-picks runs/**/best.pt when WEIGHTS is None
WEIGHTS = None
found = loader.find_yolo_checkpoints() if WEIGHTS is None else []
if found:
    print("checkpoints:", *[f"  {p.relative_to(ROOT)}" for p in found], sep="\n")
    WEIGHTS = found[0]

if WEIGHTS is None:
    print("no run/*.pt checkpoint yet - train first (cell 4) or set WEIGHTS")
else:
    loader.load_yolo(weights=WEIGHTS, image=IMAGE, conf=CONF)

---
### 6 — Serve the API + WebXR viewer

`loader.serve()` blocks, so in a notebook we start it on a background thread and print
links. In Colab, `serve_kernel_port_as_window` opens the proxy URL in a new tab
(WebXR itself needs HTTPS, so VR still wants a real tunnel — see `COLAB.md`).

In [ ]:
import threading
import time
import urllib.request

HOST, PORT = "0.0.0.0", 8000
server = threading.Thread(target=loader.serve, kwargs={"host": HOST, "port": PORT}, daemon=True)
server.start()

for _ in range(40):                      # wait for uvicorn to bind
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=1) as r:
            print("health:", r.read().decode())
            break
    except Exception:
        time.sleep(0.25)
else:
    print("server did not answer on", PORT)

print(f"\nlocal : http://127.0.0.1:{PORT}/docs")
print(f"viewer: http://127.0.0.1:{PORT}/static/viewer.html")

if IN_COLAB:
    from google.colab import output as colab_output
    colab_output.serve_kernel_port_as_window(PORT, path="/static/viewer.html")

---
### 7 — Download pretrained weights

~50 MB into `weights/` (`best.safetensors` + `config.yaml`), used by the segmenter.

In [ ]:
loader.download_weights()

---
### Equivalent CLI commands

| Notebook | Terminal |
|---|---|
| cell 3 | `python main.py check` |
| cell 4 | `python main.py train --epochs 50 --batch 16 --device 0` |
| cell 5 | `python main.py load segmenter --image <png>` |
| cell 6 | `python main.py serve --port 8000` |
| cell 7 | `python main.py download` |
| menu cell | `python main.py` |

Stop the server: *Runtime → Restart runtime* (Colab) or restart the kernel — the thread is
a daemon, so it dies with the kernel.